# Ask CIP: one plain phrase per survey question (Phase 3)

For every survey question wording: **one phrase** that completes *"65.5% of US adults ___."* and **one chart title**, so Ask CIP's answers read in plain words for any question. Written once here by an open model, never with a paid API key.

**Two passes:** the model writes each phrase (automatic checks, up to 3 attempts), then the model reviews every phrase by reading the real sentences it makes ("41.2% of US adults shopped at Walmart in the past two weeks.") and fixes any that are not natural or not faithful to the question.

**Run it (Colab Pro):**
1. *Runtime → Change runtime type → **A100 GPU*** (best; L4 or T4 also work with a smaller model, picked automatically).
2. *Runtime → Run all.* Allow Google Drive; upload `ask_phrases_input.jsonl` when asked (or put it in `MyDrive/ask-cip-colab/` first).
3. About 45–60 minutes on an A100 (32B model, download included). **Check the bottom-right corner says A100 before you run.** On a T4 it falls back to the 7B model and says so. If Colab disconnects, run all again: it continues where it stopped.

**Out** (in `MyDrive/ask-cip-colab/`, also downloaded at the end):
- `ask_phrases_out_<model>.jsonl` (e.g. `ask_phrases_out_qwen2.5-32b.jsonl`): give it to the agent: `python scripts/colab/ask_phrases.py merge <that file>` adds every reviewed phrase to `config/ask_semantic.json`.
- `ask_phrases_review_<model>.csv`: every failure, every phrase the reviewer fixed, and 100 random approved ones, for a person to check.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install -U transformers accelerate bitsandbytes

In [ ]:
import json
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive")
FOLDER = Path("/content/drive/MyDrive/ask-cip-colab")
FOLDER.mkdir(parents=True, exist_ok=True)
INPUT = FOLDER / "ask_phrases_input.jsonl"
if not INPUT.exists():
    print("Choose ask_phrases_input.jsonl")
    for name, data in files.upload().items():
        INPUT.write_bytes(data)
wordings = [json.loads(line) for line in INPUT.read_text().splitlines() if line.strip()]
print(f"{len(wordings):,} wordings")

In [ ]:
import subprocess
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

memory = int(subprocess.check_output(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"]).split()[0])
# the largest instruction model the GPU holds in 4-bit: 72B on 80 GB, 32B on an A100 40 GB, 14B on an L4, 7B on a T4
MODEL = ("Qwen/Qwen2.5-72B-Instruct" if memory >= 75000 else "Qwen/Qwen2.5-32B-Instruct" if memory >= 38000
         else "Qwen/Qwen2.5-14B-Instruct" if memory >= 22000 else "Qwen/Qwen2.5-7B-Instruct")
BATCH = 32 if memory >= 38000 else 16   # questions written at once; lower it if the GPU runs out of memory
if memory < 38000:
    print(f"NOTE: this GPU holds only {MODEL}. For the best phrases: Runtime > Change runtime type > A100 GPU, then Run all.")
short = MODEL.split("/")[1].split("-Instruct")[0].lower()   # e.g. qwen2.5-32b
OUTPUT, REVIEW = FOLDER / f"ask_phrases_out_{short}.jsonl", FOLDER / f"ask_phrases_review_{short}.csv"
print(f"{memory:,} MiB GPU memory: {MODEL}; results in {OUTPUT.name}")
tokenizer = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(MODEL, device_map="auto", quantization_config=BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16))
model.eval()

In [ ]:
import re

RULES = '''phrase: words that complete the sentence "65.5% of US adults ___." about the people who gave an answer.
- Third person, lowercase start, no "you", "your", "I", "we". Past tense for things done, present for views and plans.
- Keep the question's own time frame exactly ("in the past two weeks", "this holiday season"); never add a date or year.
- Never name a country: the same phrase is used for US adults and Chinese consumers.
- If the answers are Yes/No, the phrase describes the Yes and has no {answer}.
- Otherwise put {answer} where the chosen answer goes, exactly once. If an answer cannot fit naturally, use: said “{answer}”.
- No numbers or % of your own. At most 15 words.
title: a short chart title (at most 8 words), a finished phrase (never ending in "from", "on", "of"), capitalised, no {answer}, no %.'''

SYSTEM = '''You write plain-English phrases for a consumer survey platform. For one survey question you return JSON:
{"phrase": "...", "title": "..."}

''' + RULES + '''

Examples:
Q: Have you purchased any beauty products (whether for men or women) in the past three months? | Answers: Yes; No
{"phrase": "bought beauty products in the past three months", "title": "Bought beauty products in the past three months"}
Q: Which of the following retailers have you shopped at in the past two weeks? Select all that apply. | Answers: Walmart; Amazon; Target; Costco
{"phrase": "shopped at {answer} in the past two weeks", "title": "Retailers shopped at in the past two weeks"}
Q: How concerned are you about tariffs raising prices? | Answers: Very concerned; Somewhat concerned; Not very concerned; Not at all concerned
{"phrase": "are {answer} about tariffs raising prices", "title": "Concern about tariffs raising prices"}
Q: How much do you expect to spend on holiday gifts this year? | Answers: Less than $100; $100-$249; $250-$499; $500 or more
{"phrase": "expect to spend {answer} on holiday gifts this year", "title": "Expected holiday gift spending"}
Q: Which best describes your holiday shopping status? | Answers: I have not started; I have started but not finished; I have finished
{"phrase": "said “{answer}” about their holiday shopping", "title": "Holiday shopping status"}
Q: What is your gender? | Answers: Male; Female; Non-binary / third gender; Prefer not to say
{"phrase": "identify as {answer}", "title": "Gender"}

Return only the JSON object.'''

REVIEWER = '''You check phrases a survey platform puts into its answers. You get a survey question, its answers, a phrase, a
chart title, and sentences built from the phrase (their figures are examples; code adjusts the capital letter of an inserted
answer, so ignore that). Return only JSON:
{"ok": true} when every sentence is grammatical, natural English and says exactly what the question measured (same action,
same subject, same time frame, nothing added or lost) and the title names the same thing. Otherwise:
{"ok": false, "why": "...", "fix_phrase": "...", "fix_title": "..."} with a corrected phrase and title that follow these rules:
''' + RULES

UNSURE = re.compile(r"(not sure|unsure|don.?t know|can.?t say|cannot say|remember|recall|prefer not|maybe)", re.I)
PRONOUNS = {"you", "your", "yours", "i", "my", "me", "we", "our", "us"}
EXAMPLE_FIGURES = ("41.2", "23.5", "8.9")


def needs_answer(row):
    """A question whose answers are not just Yes/No: the phrase must say which answer."""
    answers = {a.strip().lower() for a in row["answers"]}
    return not ({"yes", "no"} <= answers and all(a in ("yes", "no") or UNSURE.search(a) for a in answers))


def problems(row, out):
    """Why a phrase or title is not usable, as a list of short reasons (empty means it passes)."""
    phrase, title = (out or {}).get("phrase", "").strip(), (out or {}).get("title", "").strip()
    if not phrase or not title:
        return ["missing phrase or title"]
    found = []
    if set(re.findall(r"[a-z']+", phrase.replace("{answer}", "").lower())) & PRONOUNS:
        found.append("uses you/I/we")
    if not phrase[0].islower():
        found.append("phrase starts with a capital")
    if "%" in phrase + title or set(re.findall(r"\d+", phrase.replace("{answer}", ""))) - set(re.findall(r"\d+", row["text"])):
        found.append("adds a number")
    if needs_answer(row) and phrase.count("{answer}") != 1:
        found.append("needs {answer} exactly once")
    if not needs_answer(row) and "{answer}" in phrase:
        found.append("Yes/No question must not have {answer}")
    if "{answer}" in title or len(title.split()) > 10 or not title[0].isupper():
        found.append("title: short, capitalised, no {answer}")
    if title.split()[-1].lower() in {"from", "on", "of", "for", "to", "at", "in", "by", "with", "about", "the", "a"}:
        found.append("title ends unfinished")
    if len(phrase.split()) > 18:
        found.append("phrase too long")
    return found


def sentences(row, phrase):
    """The sentences Ask would say with this phrase, for the reviewer to read."""
    if "{answer}" not in phrase:
        return [f"65.5% of US adults {phrase}."]
    return [f"{pct}% of US adults {phrase.replace('{answer}', a)}." for pct, a in zip(EXAMPLE_FIGURES, row["answers"])]


def question(row):
    return f"Q: {row['text']} | Answers: {'; '.join(row['answers'])}"


def writer(row, hint=""):
    """The chat for writing one phrase; a retry adds what was wrong the last time."""
    user = question(row) + (f"\nYour last answer had these problems: {hint}. Fix them." if hint else "")
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]


def reviewer(row, out):
    """The chat for reviewing one phrase through the sentences it makes."""
    lines = "\n".join(f"- {s}" for s in sentences(row, out["phrase"]))
    user = f"{question(row)}\nPhrase: {out['phrase']}\nTitle: {out['title']}\nSentences:\n{lines}"
    return [{"role": "system", "content": REVIEWER}, {"role": "user", "content": user}]


def parse(text):
    match = re.search(r"\{.*\}", text, re.S)
    try:
        return json.loads(match.group(0)) if match else None
    except json.JSONDecodeError:
        return None

print(problems({"text": "Have you bought shoes?", "answers": ["Yes", "No"]}, {"phrase": "bought shoes", "title": "Bought shoes"}))   # []

In [ ]:
CHUNK = 256   # results are saved to Drive after every CHUNK questions


def chat(conversations, sample=False):
    """The model's reply to each conversation, BATCH at a time: the most likely words, or sampled ones for a retry."""
    texts = []
    for start in range(0, len(conversations), BATCH):
        prompts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True)
                   for c in conversations[start:start + BATCH]]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=200, do_sample=sample, temperature=0.4 if sample else None,
                                 top_p=0.9 if sample else None, top_k=None, pad_token_id=tokenizer.pad_token_id)
        texts += tokenizer.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return texts


def latest():
    """The newest result per wording key, read back from the output file (so a disconnect loses nothing)."""
    rows = {}
    if OUTPUT.exists():
        for line in OUTPUT.read_text().splitlines():
            row = json.loads(line)
            rows[row["key"]] = row
    return rows


def save(rows):
    with OUTPUT.open("a") as f:
        f.writelines(json.dumps(row, ensure_ascii=False) + "\n" for row in rows)


def write(todo, attempt):
    """Writes a phrase and title for every wording in todo; attempts after the first are told what was wrong."""
    for start in range(0, len(todo), CHUNK):
        part, previous = todo[start:start + CHUNK], latest()
        hints = ["; ".join(previous.get(r["key"], {}).get("problems", [])) for r in part]
        texts = chat([writer(r, h if attempt > 1 else "") for r, h in zip(part, hints)], sample=attempt > 1)
        rows = []
        for row, text in zip(part, texts):
            out = parse(text) or {}
            found = problems(row, out)
            rows.append({"key": row["key"], "phrase": out.get("phrase", ""), "title": out.get("title", ""), "ok": not found,
                         "problems": found, "attempt": attempt, "reviewed": False, "why": "", "raw": text[:300]})
        save(rows)
        print(f"write {attempt}: {min(start + CHUNK, len(todo)):,}/{len(todo):,}")


def review(todo):
    """The reviewer approves each phrase or fixes it; a fix must pass the checks and is reviewed again next round."""
    for start in range(0, len(todo), CHUNK):
        part, previous = todo[start:start + CHUNK], latest()
        rows = []
        for row, text in zip(part, chat([reviewer(r, previous[r["key"]]) for r in part])):
            current, verdict = previous[row["key"]], parse(text)
            if not verdict:
                continue   # no readable verdict: reviewed again next round
            if verdict.get("ok") is True:
                rows.append({**current, "reviewed": True})
                continue
            fixed = {"phrase": verdict.get("fix_phrase") or "", "title": verdict.get("fix_title") or current["title"]}
            found = problems(row, fixed)
            rows.append({**current, **fixed, "ok": not found, "problems": found, "reviewed": False, "fixed_by_review": True,
                         "why": str(verdict.get("why", ""))[:300]})
        save(rows)
        print(f"review: {min(start + CHUNK, len(todo)):,}/{len(todo):,}")


done = latest()
write([r for r in wordings if r["key"] not in done], 1)
for attempt in (2, 3):
    done = latest()
    write([r for r in wordings if not done[r["key"]]["ok"] and done[r["key"]]["attempt"] < attempt], attempt)
for round_ in range(3):   # a fix is reviewed again; three rounds settle almost every phrase
    done = latest()
    review([r for r in wordings if done[r["key"]]["ok"] and not done[r["key"]]["reviewed"]])
done = latest()
approved = sum(r["ok"] and r["reviewed"] for r in done.values())
print(f"approved: {approved:,} of {len(wordings):,}")

In [ ]:
import csv, random

done = latest()
by_key = {r["key"]: r for r in wordings}
approved = [k for k, r in done.items() if r["ok"] and r["reviewed"]]
failing = [k for k, r in done.items() if not (r["ok"] and r["reviewed"])]
fixed = [k for k in approved if done[k].get("fixed_by_review")]
random.seed(7)
sample = random.sample([k for k in approved if k not in fixed], min(100, len(approved) - len(fixed)))
with REVIEW.open("w", newline="") as f:
    sheet = csv.writer(f)
    sheet.writerow(["check", "key", "question", "answers", "phrase", "title", "example sentence", "problems / reviewer's reason",
                    "correct? (yes, or the right phrase)"])
    for label, keys in (("FAILED", failing), ("fixed by reviewer", fixed), ("random approved", sample)):
        for k in keys:
            r, w = done[k], by_key[k]
            example = sentences(w, r["phrase"])[0] if r["phrase"] else ""
            sheet.writerow([label, k, w["text"], "; ".join(w["answers"]), r["phrase"], r["title"], example,
                            "; ".join(r["problems"]) or r.get("why", ""), ""])
print(f"{len(approved):,} approved ({len(fixed):,} fixed by the reviewer), {len(failing):,} failing; "
      f"review sheet: {len(failing) + len(fixed) + len(sample):,} rows")
for path in (OUTPUT, REVIEW):
    files.download(str(path))